# 05 — Scorecard

In [1]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [2]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ accepted        │
│ rejected        │
│ v_accepted_asof │
│ v_split         │
│ v_target        │
└─────────────────┘

┌─────────┬──────────────┬──────────────┐
│    n    │ min(issue_d) │ max(issue_d) │
│  int64  │     date     │     date     │
├─────────┼──────────────┼──────────────┤
│ 2260668 │ 2007-06-01   │ 2018-12-01   │
└─────────┴──────────────┴──────────────┘



In [3]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

The champion card: which attributes enter, the coefficients, the points, the odds-to-score table and the reason codes. The fit is a logistic regression on WOE; the sign of every coefficient must be negative (higher WOE = safer), and the scaling follows `config.BASE_SCORE / BASE_ODDS / PDO`.

Decisions: **I — selection rules**, **J — forced inclusions and exclusions**, **K — penalty**, then the champion is saved to `models/champion/`.

In [4]:
frame = load_frame()
tr, va, oot = (frame[frame.split == s] for s in ("train", "validate", "oot"))
specs = B.load_specs(MODELS / "bins.json"); binned = json.loads((MODELS / "binning_decisions.json").read_text(encoding="utf-8"))["binned_attributes"]
ivs = B.iv_summary(specs); order = ivs.attribute.tolist()
W = {k: B.transform_woe(d, specs) for k, d in (("train", tr), ("validate", va), ("oot", oot))}
print("train", tr.shape, "attributes available", len(order))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train (364774, 84) attributes available 19


### Decision I — Selection rules

Forward selection in IV order with a p-value to enter, a correlation guard between WOE columns, and a maximum number of attributes. Fewer attributes mean a more readable card and more stable points; more attributes mean more Gini until it stops.

**Your reasoning:**

**DRAFT** (Claude, 2026-10-31, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**Keep `p_enter=0.01`, `max_corr=0.70`, `max_attributes=15`.** Stepwise selection, taking the 19 binned attributes in IV order, enters **8**: `new_trades_share_24m`, `num_tl_op_past_12m`, `bal_ex_mort_to_limit`, `bc_open_to_buy`, `tot_hi_cred_lim`, `mths_since_recent_inq`, `mo_sin_old_rev_tl_op`, `percent_bc_gt_75`.

**Why the others were rejected:**

| Reason | Attributes |
|---|---|
| Correlation with an attribute already in | `acc_open_past_24mths` 0.73, `mo_sin_rcnt_tl` 0.70, `avg_cur_bal` 0.84, `total_bc_limit` 0.74, `tot_cur_bal` 0.96, `inq_last_6mths` 0.74, `credit_age_months` 0.87, `bc_util` 0.78 |
| Not significant | `mo_sin_rcnt_rev_tl_op` (p = 0.016) |
| Wrong sign once the others are in | `total_rev_hi_lim`, `mort_acc` |

**The fitted card:**
- All 8 coefficients are negative (right sign), from −0.377 (`mo_sin_old_rev_tl_op`) to −0.882 (`percent_bc_gt_75`), with standard errors of at most 0.044.
- VIFs are 1.12–1.72.
- McFadden R² is 0.0330.

In [5]:
SEL = dict(p_enter=0.01, max_corr=0.70, max_attributes=15)
selected, log = S.stepwise_select(W["train"], tr["bad"], order, **SEL)
display(log); print("selected:", selected)

,attribute,decision,reason
0,new_trades_share_24m,entered,"coef -1.000, p = 0"
1,acc_open_past_24mths,rejected,corr 0.73 with new_trades_share_24m
2,num_tl_op_past_12m,entered,"coef -0.496, p = 1.2e-86"
3,bal_ex_mort_to_limit,entered,"coef -0.942, p = 1.7e-310"
4,bc_open_to_buy,entered,"coef -0.851, p = 1.7e-221"
5,tot_hi_cred_lim,entered,"coef -0.439, p = 4.5e-39"
6,mo_sin_rcnt_tl,rejected,corr 0.70 with num_tl_op_past_12m
7,avg_cur_bal,rejected,corr 0.84 with tot_hi_cred_lim
8,mths_since_recent_inq,entered,"coef -0.756, p = 4.1e-159"
9,total_bc_limit,rejected,corr 0.74 with bc_open_to_buy


selected: ['new_trades_share_24m', 'num_tl_op_past_12m', 'bal_ex_mort_to_limit', 'bc_open_to_buy', 'tot_hi_cred_lim', 'mths_since_recent_inq', 'mo_sin_old_rev_tl_op', 'percent_bc_gt_75']


### Decision J — Forced in / out

An attribute the business expects on the card (delinquency history, inquiries) can be forced in even if the stepwise skipped it for correlation; one you cannot explain to a consumer is forced out.

**Your reasoning:**

**DRAFT** (Claude, 2026-10-31, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**Nothing forced in or out** (`FORCE_IN = FORCE_OUT = []`).

**`mort_acc` stays out.**
- A lender would expect to see it on the card: on its own it is safer as it rises (Gini 0.0976, IV 0.0312).
- But with the eight selected attributes in, its sign reverses. Forcing it in would put a wrong-signed attribute on the card and produce perverse reason codes.
- The same holds for `total_rev_hi_lim`.

**There is no delinquency attribute to force in.** None passed the NB03 cut, so the card's reason codes cover new accounts, balances and limits, inquiries and credit age only. This goes in the model document as a limitation.

In [6]:
FORCE_IN, FORCE_OUT = [], []
final_attrs = [a for a in selected if a not in FORCE_OUT] + [a for a in FORCE_IN if a not in selected]
card = S.fit_logistic(W["train"], tr["bad"], final_attrs)
display(S.coefficient_table(card)); display(S.vif_table(W["train"], final_attrs))
print(f"attributes {len(final_attrs)} · deviance {card.deviance:.0f} · null {card.null_deviance:.0f} · McFadden R2 {card.pseudo_r2:.4f}")

,attribute,coef,std_err,p_value,right_sign
0,new_trades_share_24m,-0.5225,0.0222,3.644063e-122,True
1,num_tl_op_past_12m,-0.6367,0.0271,1.104034e-121,True
2,bal_ex_mort_to_limit,-0.4245,0.0313,6.603588e-42,True
3,bc_open_to_buy,-0.4053,0.0332,2.656114e-34,True
4,tot_hi_cred_lim,-0.5749,0.0346,4.081998e-62,True
5,mths_since_recent_inq,-0.7884,0.0282,4.358003e-172,True
6,mo_sin_old_rev_tl_op,-0.3770,0.0315,4.528547e-33,True
7,percent_bc_gt_75,-0.8822,0.0441,5.356579e-89,True


,attribute,vif
0,new_trades_share_24m,1.63
1,num_tl_op_past_12m,1.57
2,bal_ex_mort_to_limit,1.62
3,bc_open_to_buy,1.67
4,tot_hi_cred_lim,1.72
5,mths_since_recent_inq,1.12
6,mo_sin_old_rev_tl_op,1.22
7,percent_bc_gt_75,1.62


attributes 8 · deviance 223025 · null 230626 · McFadden R2 0.0330


### Decision K — Penalty

Needed only if a coefficient is unstable or a bin was separated; an L2 penalty is a Gaussian prior on the coefficients and removes the standard errors from the fit. Default none.

**Your reasoning:**

**DRAFT** (Claude, 2026-10-31, from this notebook's full-data outputs; for review). The notebook ran with the defaults in the code cells; this draft says whether to keep them.

**No penalty** (`PENALTY = None`).

- Every coefficient is stable: standard errors 0.022–0.044, against coefficients of 0.377–0.882.
- No bin is separated, and VIFs are at most 1.72.
- A ridge penalty would only remove the standard errors.

**Results to carry to the pilot:**

| Level | Gini [95% CI] | KS | FICO midpoint Gini |
|---|---|---:|---:|
| Train | 0.2765 | — | — |
| Validate | 0.3016 | — | — |
| Out-of-time (2016) | 0.2838 [0.2789, 0.2888] | 0.2078 | 0.1805 |

**Score range.** Scores run from 451 to 647, the sum of the minimum and maximum points (450.8 and 647.5), so the card uses only part of the 300–850 scale.

**2016 calibration.** The card under-predicts 2016: in the 520–540 band the observed bad rate is 13.21% against 10.08% implied by the scaling. That is the expected level shift, which NB08 checks by band.

In [7]:
PENALTY = None                          # e.g. 1.0 for ridge
if PENALTY:
    card = S.fit_logistic(W["train"], tr["bad"], final_attrs, penalty=PENALTY); display(S.coefficient_table(card))
pts = S.points_table(card, specs); display(pts)
pts.to_csv(config.DOCS_DIR / "exhibits" / "points_table.csv", index=False)

,attribute,bin,label,woe,points,dev_share,dev_bad_rate,points_lost_vs_best
0,new_trades_share_24m,0,"(-inf, 0.05]",0.5585,83.5,0.1040,0.0572,0.0
1,new_trades_share_24m,1,"(0.05, 0.0816327]",0.4177,79.2,0.0963,0.0653,4.3
2,new_trades_share_24m,2,"(0.0816327, 0.133333]",0.2480,74.1,0.2102,0.0765,9.4
3,new_trades_share_24m,3,"(0.133333, 0.171429]",0.0759,68.9,0.1408,0.0896,14.6
4,new_trades_share_24m,4,"(0.171429, 0.2]",-0.0408,65.4,0.1072,0.0996,18.1
5,new_trades_share_24m,5,"(0.2, 0.24]",-0.1541,62.0,0.0933,0.1102,21.5
6,new_trades_share_24m,6,"(0.24, 0.266667]",-0.2491,59.1,0.0525,0.1198,24.4
7,new_trades_share_24m,7,> 0.266667,-0.4437,53.3,0.1957,0.1420,30.2
8,num_tl_op_past_12m,0,"(-inf, 0]",0.3928,81.1,0.1846,0.0669,0.0
9,num_tl_op_past_12m,1,"(0, 1]",0.1845,73.4,0.2679,0.0811,7.7


In [8]:
# Evidence — performance at each level, before any calibration work
rows = []
scored = {}
for name, d in (("in-sample (train)", tr), ("out-of-sample (validate)", va), ("out-of-time (oot)", oot)):
    sc = S.score(d, card, specs); scored[name] = sc
    m = V.delong_ci(d["bad"], sc["score"]); f = V.auc_gini_ks(d["bad"], d["fico_mid"])
    rows.append({"level": name, "n": len(d), "bad_rate": round(d["bad"].mean(), 4), "gini_card": round(m["gini"], 4),
                 "gini_ci": tuple(round(x, 4) for x in m["gini_ci"]), "ks_card": round(V.auc_gini_ks(d["bad"], sc["score"])["ks"], 4),
                 "gini_fico": round(f["gini"], 4), "score_min": int(sc.score.min()), "score_max": int(sc.score.max())})
display(pd.DataFrame(rows))
sc = scored["out-of-time (oot)"]
display(S.odds_table(sc["score"], oot["bad"], card, band=20))
display(S.reason_codes(oot.head(5), card, specs, meanings=MEANINGS).T)

,level,n,bad_rate,gini_card,gini_ci,ks_card,gini_fico,score_min,score_max
0,in-sample (train),364774,0.0960,0.2765,"(0.2705, 0.2824)",0.1986,0.1709,451,647
1,out-of-sample (validate),413791,0.1146,0.3016,"(0.2965, 0.3067)",0.2193,0.1832,451,647
2,out-of-time (oot),419611,0.1206,0.2838,"(0.2789, 0.2888)",0.2078,0.1805,451,647


,band,n,bads,goods,observed_bad_rate,observed_odds,scaled_odds,scaled_bad_rate
0,"[440, 460)",122,37,85,0.3033,2.30,2.23,0.3096
1,"[460, 480)",5147,1326,3821,0.2576,2.88,3.15,0.2410
2,"[480, 500)",32636,6995,25641,0.2143,3.67,4.46,0.1832
3,"[500, 520)",78139,13544,64595,0.1733,4.77,6.31,0.1368
4,"[520, 540)",102398,13530,88868,0.1321,6.57,8.92,0.1008
5,"[540, 560)",95101,9078,86023,0.0955,9.48,12.61,0.0735
6,"[560, 580)",63823,4264,59559,0.0668,13.97,17.84,0.0531
7,"[580, 600)",30703,1429,29274,0.0465,20.49,25.23,0.0381
8,"[600, 620)",9719,348,9371,0.0358,26.93,35.68,0.0273
9,"[620, 640)",1766,40,1726,0.0227,43.15,50.45,0.0194


,0,3,9,10,15
reason_1,Trades opened in 24 months as a share of all t...,"Months since the most recent inquiry [(-inf, 1]]","Total high credit / limit [(54892.9, 137553]]","Total high credit / limit [(137553, 190876]]",Trades opened in 24 months as a share of all t...
reason_2,Trades opened in the last 12 months [> 3],Trades opened in 24 months as a share of all t...,"Months since the most recent inquiry [(9, 16]]","Percent of bankcards above 75% utilised [(50, ...",Trades opened in the last 12 months [> 3]
reason_3,"Months since the most recent inquiry [(1, 2]]",Trades opened in the last 12 months [> 3],"Percent of bankcards above 75% utilised [(25, ...",Trades opened in 24 months as a share of all t...,"Total high credit / limit [(-inf, 39900]]"
reason_4,"Total high credit / limit [(-inf, 39900]]","Total high credit / limit [(137553, 190876]]",Non-mortgage balance over total limit [(0.2339...,"Open-to-buy on bankcards, dollars [(5335, 7967]]",Months since the oldest revolving trade opened...
reason_1_points_lost,30.2,33.9,23.5,20.1,30.2
reason_2_points_lost,29.1,30.2,13.8,18.6,29.1
reason_3_points_lost,28.5,29.1,11.5,9.4,27.5
reason_4_points_lost,27.5,20.1,10.9,8.7,15.1


In [9]:
# Save the champion
S.save_card(card, specs, MODELS / "champion")
(MODELS / "champion" / "decisions.json").write_text(json.dumps({"attributes": final_attrs, "selection": SEL, "force_in": FORCE_IN, "force_out": FORCE_OUT, "penalty": PENALTY}, indent=1), encoding="utf-8")
print("saved", MODELS / "champion")

saved C:\Users\siava\OneDrive\Desktop\score-pilot-lab\models\champion
